# Financial Report Analyst

**Project 14** — Core RAG Projects

Build a question-answering system over annual reports and earnings call transcripts.
Learn how to parse financial documents, handle tables and numerical data carefully,
and retrieve specific financial facts with citations.

**Key Skills:** Financial document parsing, table-aware chunking, numerical reasoning,
domain-specific retrieval, citation extraction.

## Project Overview

Public companies publish quarterly earnings reports, annual reports (10-K filings),
and earnings call transcripts. Analysts need to quickly find specific numbers,
compare year-over-year performance, and extract management commentary.

This notebook builds a **Financial Report Analyst** that:
1. **Parses** financial documents (narrative sections + structured tables)
2. **Chunks** content with special handling for tables and numerical data
3. **Indexes** chunks for semantic retrieval
4. **Retrieves** relevant passages and tables for financial questions
5. **Handles numerical data** carefully (exact number extraction, unit awareness)
6. **Cites sources** with document, section, and page references

### Why financial RAG is harder than general RAG
- **Tables are critical** — revenue figures, margins, and KPIs live in tables
- **Numbers need precision** — "$1.2B" vs "$1.2M" is a 1000x difference
- **Time context matters** — "revenue grew 15%" is meaningless without knowing the period
- **Domain terminology** — EBITDA, FCF, diluted EPS require domain knowledge

## Learning Goals

By the end of this notebook you will understand:
- How financial documents differ from general text for RAG
- How to parse and preserve tabular data during chunking
- Why numerical precision matters and how to handle it
- How to build table-aware retrieval for financial Q&A
- How to extract and validate numerical answers from context
- The limitations of automated financial analysis

## Problem Statement

**Scenario:** An equity analyst needs to analyze TechGlobal Inc.'s financial
performance across two years. They have the company's annual report summaries
and earnings call transcript excerpts. They want to:

- Find specific financial metrics (revenue, net income, margins)
- Compare year-over-year performance
- Extract management's forward guidance
- Understand segment-level performance
- Get answers grounded in specific document sections

**Goal:** Build a system that can accurately retrieve financial facts,
handle tabular data without losing precision, and cite sources clearly.

## Why This Project Matters

| Challenge | How Financial RAG Helps |
|-----------|----------------------|
| Reports are 100+ pages long | Retrieval finds the exact relevant section |
| Key data lives in tables, not prose | Table-aware parsing preserves structure |
| Analysts need exact numbers | Numerical extraction with validation |
| Context matters (quarter, segment, metric) | Metadata filtering narrows search |
| Manual search across many reports is slow | Semantic search spans all documents |

### Real-world applications
- **Equity research** — quickly extract KPIs from earnings reports
- **Compliance** — find specific disclosures in regulatory filings
- **Due diligence** — compare metrics across multiple companies
- **Earnings call analysis** — extract forward guidance and sentiment

## Environment Setup

In [1]:
import os
import re
import json
import hashlib
import textwrap
import warnings
from collections import defaultdict, Counter
from datetime import datetime

import numpy as np
warnings.filterwarnings("ignore")

# ── Optional dependencies ──
USE_CHROMA = False
USE_ST = False

try:
    import chromadb
    USE_CHROMA = True
    print("[OK] chromadb available")
except ImportError:
    print("[INFO] chromadb not available — using TF-IDF fallback")

try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    print("[OK] sentence-transformers available")
except ImportError:
    print("[INFO] sentence-transformers not available — using TF-IDF fallback")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\nSetup complete.")

[OK] chromadb available


[OK] sentence-transformers available

Setup complete.


In [2]:
# ── Configuration ──
CHUNK_SIZE = 600          # larger chunks for financial context
CHUNK_OVERLAP = 100
TOP_K = 5
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))

print(f"Chunk size: {CHUNK_SIZE}, Overlap: {CHUNK_OVERLAP}, Top-K: {TOP_K}")

Chunk size: 600, Overlap: 100, Top-K: 5


## Sample Financial Documents

We create realistic financial report data for a fictional company, **TechGlobal Inc.**
The corpus includes:
1. **FY2023 Annual Report Summary** — financial highlights, segment results, balance sheet
2. **FY2024 Annual Report Summary** — same structure for year-over-year comparison
3. **Q4 2024 Earnings Call Transcript** — management commentary, Q&A, forward guidance

Each document contains both **narrative text** and **structured tables** —
a critical challenge for financial RAG.

In [3]:
# ── Sample financial documents ──

FINANCIAL_DOCUMENTS = [
    {
        "doc_id": "AR-2023",
        "title": "TechGlobal Inc. Annual Report FY2023",
        "doc_type": "annual_report",
        "company": "TechGlobal Inc.",
        "period": "FY2023",
        "date": "2024-02-15",
        "sections": [
            {
                "section_id": "AR2023-S1",
                "heading": "Financial Highlights",
                "content_type": "narrative",
                "text": """TechGlobal Inc. delivered strong financial results for fiscal year 2023, demonstrating resilient growth across all business segments despite macroeconomic headwinds. Total revenue reached $12.4 billion, representing a 14% increase from the prior year's $10.9 billion. The growth was driven primarily by our Cloud Services division, which saw 28% year-over-year revenue growth.

Operating income expanded to $2.8 billion, up from $2.3 billion in FY2022, reflecting a 22% improvement. Operating margin improved to 22.6% from 21.1%, benefiting from operational efficiencies and favorable product mix shift toward higher-margin cloud subscriptions.

Net income attributable to shareholders was $2.1 billion, or $4.20 per diluted share, compared to $1.7 billion, or $3.40 per diluted share, in the prior year. Free cash flow generation remained robust at $3.2 billion, enabling continued investment in R&D and strategic acquisitions."""
            },
            {
                "section_id": "AR2023-S2",
                "heading": "Revenue by Segment",
                "content_type": "table",
                "text": """Revenue by Business Segment (in millions USD):

| Segment | FY2023 Revenue | FY2022 Revenue | YoY Growth |
|---------|---------------|---------------|------------|
| Cloud Services | $5,580 | $4,360 | 28.0% |
| Enterprise Software | $3,720 | $3,480 | 6.9% |
| Professional Services | $2,170 | $2,050 | 5.9% |
| Hardware Solutions | $930 | $1,010 | -7.9% |
| Total | $12,400 | $10,900 | 13.8% |

Cloud Services accounted for 45% of total revenue in FY2023, up from 40% in FY2022. Enterprise Software remained the second-largest segment at 30% of revenue. Hardware Solutions continued its planned decline as the company transitions to software-defined solutions."""
            },
            {
                "section_id": "AR2023-S3",
                "heading": "Profitability Metrics",
                "content_type": "table",
                "text": """Key Profitability Metrics:

| Metric | FY2023 | FY2022 | Change |
|--------|--------|--------|--------|
| Gross Profit | $8,060M | $6,976M | +15.5% |
| Gross Margin | 65.0% | 64.0% | +1.0pp |
| Operating Income | $2,800M | $2,299M | +21.8% |
| Operating Margin | 22.6% | 21.1% | +1.5pp |
| Net Income | $2,100M | $1,700M | +23.5% |
| Net Margin | 16.9% | 15.6% | +1.3pp |
| EBITDA | $3,500M | $2,950M | +18.6% |
| EBITDA Margin | 28.2% | 27.1% | +1.1pp |
| Diluted EPS | $4.20 | $3.40 | +23.5% |
| Free Cash Flow | $3,200M | $2,700M | +18.5% |

All margins showed improvement year-over-year, reflecting the company's successful transition toward higher-margin recurring revenue streams and disciplined cost management."""
            },
            {
                "section_id": "AR2023-S4",
                "heading": "Balance Sheet Summary",
                "content_type": "table",
                "text": """Condensed Balance Sheet (as of December 31, 2023):

| Item | FY2023 | FY2022 |
|------|--------|--------|
| Cash and Equivalents | $4,200M | $3,500M |
| Short-term Investments | $1,800M | $1,500M |
| Total Current Assets | $8,100M | $7,200M |
| Total Assets | $22,500M | $20,100M |
| Total Current Liabilities | $5,400M | $4,800M |
| Long-term Debt | $4,000M | $4,500M |
| Total Stockholders' Equity | $11,800M | $9,900M |
| Debt-to-Equity Ratio | 0.34 | 0.45 |

The company reduced long-term debt by $500M during FY2023 while growing cash reserves by $700M. The debt-to-equity ratio improved from 0.45 to 0.34, strengthening the balance sheet significantly."""
            },
            {
                "section_id": "AR2023-S5",
                "heading": "Research and Development",
                "content_type": "narrative",
                "text": """R&D investment totaled $1.86 billion in FY2023, representing 15.0% of revenue, compared to $1.53 billion (14.0% of revenue) in FY2022. Key R&D focus areas included:

Artificial Intelligence and Machine Learning: The company invested $620 million in AI/ML capabilities, including developing proprietary large language models for enterprise applications, computer vision solutions for manufacturing quality control, and predictive analytics platforms.

Cloud Infrastructure: $480 million was allocated to next-generation cloud infrastructure, including edge computing capabilities, serverless architecture improvements, and multi-cloud management tools.

Cybersecurity: $340 million was invested in advanced threat detection using AI, zero-trust architecture implementation, and quantum-resistant encryption research.

The company filed 342 new patents in FY2023 and maintained a total patent portfolio of 4,200 active patents globally."""
            },
        ]
    },
    {
        "doc_id": "AR-2024",
        "title": "TechGlobal Inc. Annual Report FY2024",
        "doc_type": "annual_report",
        "company": "TechGlobal Inc.",
        "period": "FY2024",
        "date": "2025-02-14",
        "sections": [
            {
                "section_id": "AR2024-S1",
                "heading": "Financial Highlights",
                "content_type": "narrative",
                "text": """TechGlobal Inc. achieved record financial performance in fiscal year 2024, with total revenue of $14.9 billion, representing 20% year-over-year growth from FY2023's $12.4 billion. This acceleration was fueled by exceptional demand for AI-powered cloud solutions and successful expansion into new geographic markets.

Operating income reached $3.6 billion, a 29% increase from $2.8 billion in FY2023. Operating margin expanded to 24.2%, up from 22.6%, driven by AI product premiums and continued operational leverage.

Net income grew to $2.7 billion, or $5.40 per diluted share, compared to $2.1 billion, or $4.20 per diluted share, in FY2023 — a 29% increase in earnings per share. Free cash flow reached $4.1 billion, providing ample capacity for both organic investment and M&A activity.

The company completed two strategic acquisitions: DataSphere (cloud data analytics, $1.2B) and SecureNet (AI-powered cybersecurity, $800M), which are expected to contribute $600M in incremental revenue in FY2025."""
            },
            {
                "section_id": "AR2024-S2",
                "heading": "Revenue by Segment",
                "content_type": "table",
                "text": """Revenue by Business Segment (in millions USD):

| Segment | FY2024 Revenue | FY2023 Revenue | YoY Growth |
|---------|---------------|---------------|------------|
| Cloud Services | $7,450 | $5,580 | 33.5% |
| Enterprise Software | $4,030 | $3,720 | 8.3% |
| Professional Services | $2,520 | $2,170 | 16.1% |
| Hardware Solutions | $900 | $930 | -3.2% |
| Total | $14,900 | $12,400 | 20.2% |

Cloud Services now represents 50% of total revenue, crossing the majority threshold for the first time. AI-powered products within Cloud Services grew 85% year-over-year and now represent $2.2 billion in annual recurring revenue. Professional Services growth accelerated due to increased demand for AI implementation consulting."""
            },
            {
                "section_id": "AR2024-S3",
                "heading": "Profitability Metrics",
                "content_type": "table",
                "text": """Key Profitability Metrics:

| Metric | FY2024 | FY2023 | Change |
|--------|--------|--------|--------|
| Gross Profit | $10,132M | $8,060M | +25.7% |
| Gross Margin | 68.0% | 65.0% | +3.0pp |
| Operating Income | $3,600M | $2,800M | +28.6% |
| Operating Margin | 24.2% | 22.6% | +1.6pp |
| Net Income | $2,700M | $2,100M | +28.6% |
| Net Margin | 18.1% | 16.9% | +1.2pp |
| EBITDA | $4,500M | $3,500M | +28.6% |
| EBITDA Margin | 30.2% | 28.2% | +2.0pp |
| Diluted EPS | $5.40 | $4.20 | +28.6% |
| Free Cash Flow | $4,100M | $3,200M | +28.1% |

Gross margin expansion to 68% reflects the growing mix of high-margin AI and cloud subscription revenue. EBITDA margin crossed 30% for the first time in company history."""
            },
            {
                "section_id": "AR2024-S4",
                "heading": "Geographic Revenue Breakdown",
                "content_type": "table",
                "text": """Revenue by Geography (in millions USD):

| Region | FY2024 Revenue | FY2023 Revenue | YoY Growth |
|--------|---------------|---------------|------------|
| North America | $8,940 | $7,688 | 16.3% |
| Europe | $3,278 | $2,604 | 25.9% |
| Asia-Pacific | $2,086 | $1,612 | 29.4% |
| Rest of World | $596 | $496 | 20.2% |
| Total | $14,900 | $12,400 | 20.2% |

International revenue grew faster than domestic, reaching 40% of total revenue versus 38% in FY2023. Asia-Pacific was the fastest-growing region, driven by cloud adoption in Japan, India, and Southeast Asia. The company opened new data centers in Singapore and Frankfurt to support international expansion."""
            },
            {
                "section_id": "AR2024-S5",
                "heading": "Workforce and Operations",
                "content_type": "narrative",
                "text": """Total headcount at year-end was 52,000 employees, up from 45,000 in FY2023. The increase includes approximately 3,000 employees from the DataSphere and SecureNet acquisitions.

Revenue per employee increased to $286,500 from $275,600, reflecting improved operational efficiency. Employee retention rate was 91%, above the industry average of 85%.

The company invested $180 million in employee development programs, including AI skills training for 15,000 existing employees. Stock-based compensation expense was $890 million, representing 6.0% of revenue."""
            },
        ]
    },
    {
        "doc_id": "EC-Q4-2024",
        "title": "TechGlobal Inc. Q4 2024 Earnings Call Transcript",
        "doc_type": "earnings_call",
        "company": "TechGlobal Inc.",
        "period": "Q4-2024",
        "date": "2025-01-28",
        "sections": [
            {
                "section_id": "EC-S1",
                "heading": "CEO Opening Remarks",
                "content_type": "narrative",
                "text": """Thank you, and good afternoon everyone. I'm pleased to report that TechGlobal delivered an exceptional fourth quarter, capping off a record year. Q4 revenue was $4.1 billion, up 22% year-over-year, with Cloud Services growing 35% in the quarter.

Our AI-powered product portfolio continues to see strong adoption. We signed over 200 new enterprise AI deals in Q4 alone, with an average contract value of $2.8 million, up 40% from Q4 last year. Our AI annual recurring revenue exited the year at $2.2 billion, and we see a clear path to $3.5 billion by the end of FY2025.

Customer retention rates remained exceptional at 97% for cloud subscriptions, with net revenue retention at 118%, meaning existing customers are expanding their spend by 18% on average. This demonstrates the stickiness of our platform and the value customers are deriving from our solutions."""
            },
            {
                "section_id": "EC-S2",
                "heading": "CFO Financial Review",
                "content_type": "narrative",
                "text": """Thank you, Sarah. Let me walk through the Q4 and full-year financial details.

Q4 revenue of $4.1 billion exceeded our guidance range of $3.9 to $4.0 billion. Full-year revenue was $14.9 billion, up 20% year-over-year. On a constant currency basis, growth was approximately 21%.

Q4 gross margin was 69.2%, up from 65.8% in Q4 2023. The improvement reflects higher cloud mix, AI product premiums, and continued infrastructure optimization. Full-year gross margin was 68.0%.

Q4 operating margin was 25.1%, compared to 23.0% in Q4 last year. Full-year operating margin was 24.2%. We generated $1.2 billion in free cash flow in Q4, bringing full-year FCF to $4.1 billion, up 28% year-over-year.

Regarding capital allocation: we repurchased $500 million of shares during Q4, bringing full-year buybacks to $1.5 billion. The Board has approved a new $3 billion share repurchase authorization. We also increased the quarterly dividend by 12% to $0.56 per share."""
            },
            {
                "section_id": "EC-S3",
                "heading": "Forward Guidance",
                "content_type": "narrative",
                "text": """For FY2025, we are providing the following guidance:

Total revenue: $17.5 billion to $18.0 billion, representing 17-21% year-over-year growth. This includes approximately $600 million in expected contribution from the DataSphere and SecureNet acquisitions.

Operating margin: 25.0% to 26.0%, reflecting continued leverage and mix improvement. We expect gross margin to reach approximately 69-70%.

Diluted EPS: $6.30 to $6.60, representing 17-22% growth from FY2024's $5.40.

Free cash flow: greater than $4.5 billion.

Capital expenditures: $2.0 billion to $2.2 billion, primarily for data center expansion and AI infrastructure buildout. We plan to open 3 new data centers in 2025.

For Q1 2025 specifically, we expect revenue of $4.2 billion to $4.3 billion and operating margin of approximately 24.5%."""
            },
            {
                "section_id": "EC-S4",
                "heading": "Analyst Q&A — AI Strategy",
                "content_type": "narrative",
                "text": """Analyst (Morgan Stanley): Can you give us more color on the AI revenue composition and what's driving the acceleration?

CEO: Great question. Our AI revenue of $2.2 billion breaks down roughly as follows: about 55% comes from our AI-as-a-Service platform, which is essentially our managed AI infrastructure and model hosting. About 30% comes from AI-embedded features within our existing Enterprise Software products — things like intelligent automation, predictive analytics, and AI-powered security. The remaining 15% comes from our Professional Services AI consulting practice.

The acceleration is driven by three factors. First, enterprises are past the experimentation phase and are now deploying AI in production at scale. Second, our proprietary models are showing competitive performance with significantly lower inference costs than alternatives. Third, the data moat effect — the more data customers process through our platform, the better our models perform for their specific use case, creating a flywheel effect.

We expect AI revenue to reach $3.5 billion in FY2025, representing approximately 60% year-over-year growth. Longer term, we see AI becoming a $10 billion business for TechGlobal by 2028."""
            },
            {
                "section_id": "EC-S5",
                "heading": "Analyst Q&A — Competitive Landscape",
                "content_type": "narrative",
                "text": """Analyst (Goldman Sachs): How do you see the competitive landscape evolving, particularly with the large hyperscalers investing heavily in AI?

CEO: We actually see the hyperscaler investments as tailwinds for us, not headwinds. Our customers are increasingly adopting multi-cloud strategies, and our platform is designed to work across AWS, Azure, and GCP. We don't compete with the infrastructure layer — we compete at the application and platform layer, which is where the differentiation and margin are.

What sets us apart is our vertical expertise. We've built industry-specific AI solutions for healthcare, financial services, and manufacturing that the hyperscalers simply don't offer. Our healthcare AI platform, for example, is used by 8 of the top 10 hospital systems in the US.

The other differentiator is data privacy. Many of our enterprise customers, particularly in regulated industries, want to run AI models on their own data without it leaving their environment. Our on-premise and hybrid deployment options address this need directly. About 40% of our AI deployments are on-premise or hybrid, which is a segment the pure-cloud players struggle to serve."""
            },
        ]
    },
]

print(f"Created {len(FINANCIAL_DOCUMENTS)} financial documents:\n")
for doc in FINANCIAL_DOCUMENTS:
    section_count = len(doc["sections"])
    tables = sum(1 for s in doc["sections"] if s["content_type"] == "table")
    total_chars = sum(len(s["text"]) for s in doc["sections"])
    print(f"  [{doc['doc_id']}] {doc['title']}")
    print(f"    Period: {doc['period']} | Sections: {section_count} "
          f"({tables} tables) | {total_chars:,} chars")


Created 3 financial documents:

  [AR-2023] TechGlobal Inc. Annual Report FY2023
    Period: FY2023 | Sections: 5 (3 tables) | 3,898 chars
  [AR-2024] TechGlobal Inc. Annual Report FY2024
    Period: FY2024 | Sections: 5 (3 tables) | 3,661 chars
  [EC-Q4-2024] TechGlobal Inc. Q4 2024 Earnings Call Transcript
    Period: Q4-2024 | Sections: 5 (0 tables) | 5,014 chars


## Handling Tables and Numbers in Financial Documents

This is the **critical differentiator** between financial RAG and general-purpose RAG.

### The Table Problem

Standard text chunking destroys table structure. If you split this table row:

```
| Cloud Services | $5,580 | $4,360 | 28.0% |
```

across two chunks, one chunk gets the segment name without numbers, and the other
gets numbers without context. **Both chunks become useless.**

### Our Approach: Table-Aware Chunking

1. **Detect tables** — identify sections containing markdown tables or structured data
2. **Keep tables intact** — never split a table row across chunks
3. **Attach table headers** — every table chunk includes column headers
4. **Tag content type** — mark chunks as "table" or "narrative" in metadata
5. **Preserve units** — keep currency symbols, percentages, and units with numbers

### The Number Precision Problem

Financial Q&A requires exact numbers, not paraphrased approximations:
- ✅ "Revenue was $14.9 billion in FY2024"
- ❌ "Revenue was approximately $15 billion"
- ❌ "Revenue was in the billions"

Our retriever returns exact text from source documents rather than
generating approximate answers.

In [4]:
# ── Financial document parser ──

class FinancialSection:
    """Represents a parsed section of a financial document."""

    def __init__(self, section_dict, doc_meta):
        self.section_id = section_dict["section_id"]
        self.heading = section_dict["heading"]
        self.content_type = section_dict["content_type"]  # "narrative" or "table"
        self.text = section_dict["text"].strip()
        self.doc_id = doc_meta["doc_id"]
        self.doc_title = doc_meta["title"]
        self.doc_type = doc_meta["doc_type"]
        self.company = doc_meta["company"]
        self.period = doc_meta["period"]
        self.date = doc_meta["date"]
        self.has_table = "|" in self.text and "---" in self.text
        self.numbers = self._extract_numbers()

    def _extract_numbers(self):
        """Extract financial numbers with context."""
        patterns = [
            r'\$[\d,.]+\s*(?:billion|million|B|M)?',
            r'[\d,.]+%',
            r'\$[\d,.]+',
        ]
        numbers = []
        for p in patterns:
            numbers.extend(re.findall(p, self.text, re.IGNORECASE))
        return numbers

    @property
    def metadata(self):
        return {
            "section_id": self.section_id,
            "heading": self.heading,
            "content_type": self.content_type,
            "doc_id": self.doc_id,
            "doc_title": self.doc_title,
            "doc_type": self.doc_type,
            "period": self.period,
            "date": self.date,
            "has_table": str(self.has_table),
            "number_count": len(self.numbers),
        }


# Parse all sections
all_sections = []
for doc in FINANCIAL_DOCUMENTS:
    meta = {k: doc[k] for k in ["doc_id", "title", "doc_type", "company", "period", "date"]}
    for sec_dict in doc["sections"]:
        section = FinancialSection(sec_dict, meta)
        all_sections.append(section)

print(f"Parsed {len(all_sections)} sections:\n")
for sec in all_sections:
    table_tag = " [TABLE]" if sec.has_table else ""
    print(f"  [{sec.doc_id}] {sec.heading}{table_tag} "
          f"({len(sec.text)} chars, {len(sec.numbers)} numbers)")

Parsed 15 sections:

  [AR-2023] Financial Highlights (928 chars, 23 numbers)
  [AR-2023] Revenue by Segment [TABLE] (660 chars, 28 numbers)
  [AR-2023] Profitability Metrics [TABLE] (718 chars, 38 numbers)
  [AR-2023] Balance Sheet Summary [TABLE] (658 chars, 32 numbers)
  [AR-2023] Research and Development (934 chars, 12 numbers)
  [AR-2024] Financial Highlights (1004 chars, 29 numbers)
  [AR-2024] Revenue by Segment [TABLE] (722 chars, 29 numbers)
  [AR-2024] Profitability Metrics [TABLE] (715 chars, 40 numbers)
  [AR-2024] Geographic Revenue Breakdown [TABLE] (664 chars, 27 numbers)
  [AR-2024] Workforce and Operations (556 chars, 11 numbers)
  [EC-Q4-2024] CEO Opening Remarks (863 chars, 14 numbers)
  [EC-Q4-2024] CFO Financial Review (957 chars, 30 numbers)
  [EC-Q4-2024] Forward Guidance (806 chars, 28 numbers)
  [EC-Q4-2024] Analyst Q&A — AI Strategy (1215 chars, 10 numbers)
  [EC-Q4-2024] Analyst Q&A — Competitive Landscape (1173 chars, 1 numbers)


## Table-Aware Chunking

Our chunker applies different strategies based on content type:
- **Narrative text**: standard sliding-window chunks with overlap
- **Table sections**: keep tables whole or split by row groups, always
  preserving headers

This ensures that financial data remains interpretable in every chunk.

In [5]:
# ── Table-aware chunking ──

class FinancialChunk:
    """A chunk of financial text with metadata and content type."""

    def __init__(self, text, section, chunk_idx, total_chunks):
        self.text = text
        self.section_id = section.section_id
        self.heading = section.heading
        self.content_type = section.content_type
        self.doc_id = section.doc_id
        self.doc_title = section.doc_title
        self.doc_type = section.doc_type
        self.period = section.period
        self.date = section.date
        self.has_table = "|" in text and ("---" in text or "|" in text)
        self.chunk_idx = chunk_idx
        self.total_chunks = total_chunks
        self.numbers = self._extract_numbers()

        content_hash = hashlib.md5(
            f"{self.doc_id}:{self.section_id}:{chunk_idx}".encode()
        ).hexdigest()[:8]
        self.chunk_id = f"{self.doc_id}-{self.section_id}-{content_hash}"

    def _extract_numbers(self):
        patterns = [r'\$[\d,.]+\s*(?:billion|million|B|M)?', r'[\d,.]+%', r'\$[\d,.]+']
        nums = []
        for p in patterns:
            nums.extend(re.findall(p, self.text, re.IGNORECASE))
        return nums

    @property
    def metadata(self):
        return {
            "chunk_id": self.chunk_id,
            "section_id": self.section_id,
            "heading": self.heading,
            "content_type": self.content_type,
            "doc_id": self.doc_id,
            "doc_title": self.doc_title,
            "doc_type": self.doc_type,
            "period": self.period,
            "date": self.date,
            "has_table": str(self.has_table),
            "number_count": len(self.numbers),
        }

    @property
    def citation(self):
        tbl = " [TABLE]" if self.has_table else ""
        return (f"[{self.doc_id}] {self.heading}{tbl} "
                f"(chunk {self.chunk_idx+1}/{self.total_chunks})")


def chunk_section(section, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Chunk a financial section with table awareness."""
    text = section.text

    # If section fits in one chunk, keep it whole
    if len(text) <= chunk_size:
        return [FinancialChunk(text, section, 0, 1)]

    # Table sections: try to keep table intact or split by row groups
    if section.has_table:
        return _chunk_table_section(text, section, chunk_size)

    # Narrative sections: standard sliding window
    return _chunk_narrative(text, section, chunk_size, overlap)


def _chunk_table_section(text, section, chunk_size):
    """Split table sections while preserving headers."""
    lines = text.split("\n")

    # Find table header and separator
    header_lines = []
    table_rows = []
    pre_table = []
    post_table = []
    in_table = False
    past_table = False

    for line in lines:
        stripped = line.strip()
        if not in_table and not past_table:
            if "|" in stripped and "---" not in stripped and len(stripped) > 5:
                in_table = True
                header_lines.append(line)
            else:
                pre_table.append(line)
        elif in_table:
            if "---" in stripped:
                header_lines.append(line)
            elif "|" in stripped and len(stripped) > 5:
                table_rows.append(line)
            else:
                if stripped:
                    past_table = True
                    in_table = False
                    post_table.append(line)
        else:
            post_table.append(line)

    # Combine pre-table text + header
    header_block = "\n".join(pre_table + header_lines).strip()
    post_block = "\n".join(post_table).strip()

    # If everything fits in one chunk, return as-is
    if len(text) <= chunk_size:
        return [FinancialChunk(text, section, 0, 1)]

    # Split table rows into groups that fit within chunk_size
    chunks = []
    current_rows = []
    current_size = len(header_block) + 2  # +2 for newlines

    for row in table_rows:
        if current_size + len(row) + 1 > chunk_size and current_rows:
            chunk_text = header_block + "\n" + "\n".join(current_rows)
            chunks.append(chunk_text.strip())
            current_rows = []
            current_size = len(header_block) + 2
        current_rows.append(row)
        current_size += len(row) + 1

    if current_rows:
        chunk_text = header_block + "\n" + "\n".join(current_rows)
        if post_block:
            chunk_text += "\n\n" + post_block
        chunks.append(chunk_text.strip())

    if not chunks:
        chunks = [text]

    return [FinancialChunk(ct, section, i, len(chunks)) for i, ct in enumerate(chunks)]


def _chunk_narrative(text, section, chunk_size, overlap):
    """Standard narrative chunking with overlap."""
    parts = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        parts.append(text[start:end].strip())
        start += chunk_size - overlap

    return [FinancialChunk(p, section, i, len(parts)) for i, p in enumerate(parts) if p]


# Chunk all sections
all_chunks = []
for section in all_sections:
    section_chunks = chunk_section(section)
    all_chunks.extend(section_chunks)

print(f"Total chunks: {len(all_chunks)}\n")
for chunk in all_chunks:
    table_tag = " [TABLE]" if chunk.has_table else ""
    print(f"  {chunk.chunk_id}: {len(chunk.text)} chars, "
          f"{len(chunk.numbers)} numbers{table_tag} | {chunk.heading}")

Total chunks: 26

  AR-2023-AR2023-S1-535b708f: 600 chars, 13 numbers | Financial Highlights
  AR-2023-AR2023-S1-b80222ec: 428 chars, 12 numbers | Financial Highlights
  AR-2023-AR2023-S2-fffcc5ab: 660 chars, 28 numbers [TABLE] | Revenue by Segment
  AR-2023-AR2023-S3-b32de698: 718 chars, 38 numbers [TABLE] | Profitability Metrics
  AR-2023-AR2023-S4-3df72610: 658 chars, 32 numbers [TABLE] | Balance Sheet Summary
  AR-2023-AR2023-S5-a4e500e7: 600 chars, 10 numbers | Research and Development
  AR-2023-AR2023-S5-fe2c8510: 434 chars, 2 numbers | Research and Development
  AR-2024-AR2024-S1-65ed117a: 600 chars, 18 numbers | Financial Highlights
  AR-2024-AR2024-S1-69392017: 504 chars, 17 numbers | Financial Highlights
  AR-2024-AR2024-S1-221b896d: 4 chars, 0 numbers | Financial Highlights
  AR-2024-AR2024-S2-c557c188: 722 chars, 29 numbers [TABLE] | Revenue by Segment
  AR-2024-AR2024-S3-13083ed0: 715 chars, 40 numbers [TABLE] | Profitability Metrics
  AR-2024-AR2024-S4-53549136: 664 chars

In [6]:
# ── Verify table preservation ──

table_chunks = [c for c in all_chunks if c.has_table]
narrative_chunks = [c for c in all_chunks if not c.has_table]

print(f"Table chunks: {len(table_chunks)}")
print(f"Narrative chunks: {len(narrative_chunks)}")

print(f"\n--- Sample table chunk ---")
sample_table = table_chunks[0]
print(f"Chunk ID: {sample_table.chunk_id}")
print(f"Source: {sample_table.citation}")
print(f"Numbers found: {sample_table.numbers[:8]}")
print(f"\nText:\n{sample_table.text[:500]}")

Table chunks: 6
Narrative chunks: 20

--- Sample table chunk ---
Chunk ID: AR-2023-AR2023-S2-fffcc5ab
Source: [AR-2023] Revenue by Segment [TABLE] (chunk 1/1)
Numbers found: ['$5,580 ', '$4,360 ', '$3,720 ', '$3,480 ', '$2,170 ', '$2,050 ', '$930 ', '$1,010 ']

Text:
Revenue by Business Segment (in millions USD):

| Segment | FY2023 Revenue | FY2022 Revenue | YoY Growth |
|---------|---------------|---------------|------------|
| Cloud Services | $5,580 | $4,360 | 28.0% |
| Enterprise Software | $3,720 | $3,480 | 6.9% |
| Professional Services | $2,170 | $2,050 | 5.9% |
| Hardware Solutions | $930 | $1,010 | -7.9% |
| Total | $12,400 | $10,900 | 13.8% |

Cloud Services accounted for 45% of total revenue in FY2023, up from 40% in FY2022. Enterprise Software r


## Building the Financial Index

We index all chunks with their metadata, enabling filtered search by:
- **Period** — only FY2024, or only Q4 2024
- **Document type** — annual reports vs earnings calls
- **Content type** — tables vs narrative text

In [7]:
# ── Build financial index ──

class FinancialIndex:
    """Vector index for financial document retrieval."""

    def __init__(self, chunks):
        self.chunks = chunks
        self.texts = [c.text for c in chunks]
        self.chunk_map = {c.chunk_id: c for c in chunks}

        if USE_ST and USE_CHROMA:
            self._build_chroma()
        else:
            self._build_tfidf()

    def _build_chroma(self):
        print("Building index with sentence-transformers + ChromaDB...")
        self.model = SentenceTransformer(EMBEDDING_MODEL)
        self.client = chromadb.Client()
        try:
            self.client.delete_collection("financial")
        except Exception:
            pass
        self.collection = self.client.create_collection(
            name="financial", metadata={"hnsw:space": "cosine"}
        )
        embeddings = self.model.encode(self.texts, show_progress_bar=False).tolist()
        ids = [c.chunk_id for c in self.chunks]
        metas = [c.metadata for c in self.chunks]
        self.collection.add(ids=ids, embeddings=embeddings,
                            documents=self.texts, metadatas=metas)
        self.backend = "chroma"
        print(f"Indexed {self.collection.count()} chunks")

    def _build_tfidf(self):
        print("Building TF-IDF fallback index...")
        self.vectorizer = TfidfVectorizer(
            max_features=5000, stop_words="english", ngram_range=(1, 2)
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        self.backend = "tfidf"
        print(f"Indexed {self.tfidf_matrix.shape[0]} chunks")

    def search(self, query, top_k=TOP_K, period_filter=None,
               doc_type_filter=None, table_only=False):
        """Search for financial chunks with optional filters."""
        if self.backend == "chroma":
            return self._search_chroma(query, top_k, period_filter,
                                        doc_type_filter, table_only)
        return self._search_tfidf(query, top_k, period_filter,
                                   doc_type_filter, table_only)

    def _search_chroma(self, query, top_k, period_filter, doc_type_filter, table_only):
        conditions = []
        if period_filter:
            conditions.append({"period": period_filter})
        if doc_type_filter:
            conditions.append({"doc_type": doc_type_filter})
        if table_only:
            conditions.append({"has_table": "True"})
        where = None
        if len(conditions) == 1:
            where = conditions[0]
        elif len(conditions) > 1:
            where = {"$and": conditions}

        query_emb = self.model.encode([query]).tolist()
        results = self.collection.query(
            query_embeddings=query_emb,
            n_results=min(top_k, self.collection.count()),
            where=where,
        )
        output = []
        for i, uid in enumerate(results["ids"][0]):
            chunk = self.chunk_map[uid]
            sim = 1.0 - results["distances"][0][i]
            output.append((chunk, sim))
        return output

    def _search_tfidf(self, query, top_k, period_filter, doc_type_filter, table_only):
        candidates = []
        for i, c in enumerate(self.chunks):
            if period_filter and c.period != period_filter:
                continue
            if doc_type_filter and c.doc_type != doc_type_filter:
                continue
            if table_only and not c.has_table:
                continue
            candidates.append(i)
        if not candidates:
            return []
        query_vec = self.vectorizer.transform([query])
        cand_matrix = self.tfidf_matrix[candidates]
        sims = cosine_similarity(query_vec, cand_matrix).flatten()
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.chunks[candidates[i]], float(sims[i])) for i in top_idx]


fin_index = FinancialIndex(all_chunks)

Building index with sentence-transformers + ChromaDB...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Indexed 26 chunks


## Baseline: Keyword Search

In [8]:
# ── Baseline ──

def keyword_search(query, chunks, top_k=TOP_K):
    q_terms = set(query.lower().split())
    scored = []
    for c in chunks:
        hits = sum(1 for t in q_terms if t in c.text.lower())
        scored.append((c, hits / max(len(q_terms), 1)))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

test_q = "What was TechGlobal's revenue in FY2024?"
baseline = keyword_search(test_q, all_chunks, top_k=3)

print(f"Query: {test_q}\n")
for c, score in baseline:
    print(f"  Score: {score:.2f} | {c.citation}")
    print(f"  Numbers: {c.numbers[:5]}")
    print()

Query: What was TechGlobal's revenue in FY2024?

  Score: 0.50 | [AR-2023] Financial Highlights (chunk 1/2)
  Numbers: ['$12.4 billion', '$10.9 billion', '$2.8 billion', '$2.3 billion', '14%']

  Score: 0.50 | [AR-2023] Research and Development (chunk 1/2)
  Numbers: ['$1.86 billion', '$1.53 billion', '$620 million', '$480 million', '15.0%']

  Score: 0.50 | [AR-2024] Financial Highlights (chunk 1/3)
  Numbers: ['$14.9 billion', '$12.4 billion', '$3.6 billion', '$2.8 billion', '$2.7 billion']



## Semantic Retrieval

In [9]:
# ── Semantic retrieval ──

test_q = "What was TechGlobal's revenue in FY2024?"
results = fin_index.search(test_q, top_k=3)

print(f"Query: {test_q}\n")
for c, score in results:
    table_tag = " [TABLE]" if c.has_table else ""
    print(f"  Score: {score:.3f} | {c.citation}")
    print(f"  Period: {c.period} | Type: {c.content_type}{table_tag}")
    print(f"  Numbers: {c.numbers[:5]}")
    print(f"  Preview: {c.text[:120]}...")
    print()

Query: What was TechGlobal's revenue in FY2024?

  Score: 0.717 | [AR-2023] Financial Highlights (chunk 1/2)
  Period: FY2023 | Type: narrative
  Numbers: ['$12.4 billion', '$10.9 billion', '$2.8 billion', '$2.3 billion', '14%']
  Preview: TechGlobal Inc. delivered strong financial results for fiscal year 2023, demonstrating resilient growth across all busin...

  Score: 0.717 | [AR-2024] Financial Highlights (chunk 1/3)
  Period: FY2024 | Type: narrative
  Numbers: ['$14.9 billion', '$12.4 billion', '$3.6 billion', '$2.8 billion', '$2.7 billion']
  Preview: TechGlobal Inc. achieved record financial performance in fiscal year 2024, with total revenue of $14.9 billion, represen...

  Score: 0.632 | [EC-Q4-2024] Forward Guidance (chunk 1/2)
  Period: Q4-2024 | Type: narrative
  Numbers: ['$17.5 billion', '$18.0 billion', '$600 million', '$6.30 ', '$6.60, ']
  Preview: For FY2025, we are providing the following guidance:

Total revenue: $17.5 billion to $18.0 billion, representing 17-21%.

In [10]:
# ── Financial topic queries ──

fin_queries = [
    "What was the operating margin in FY2024?",
    "How fast is Cloud Services revenue growing?",
    "What is the company's AI revenue?",
    "What guidance did management give for FY2025?",
    "How much was spent on R&D?",
    "What is the debt-to-equity ratio?",
]

print("=== Multi-Topic Financial Retrieval ===\n")
for q in fin_queries:
    results = fin_index.search(q, top_k=1)
    if results:
        c, score = results[0]
        print(f"Q: {q}")
        print(f"  [{score:.3f}] {c.citation} ({c.period})")
        print(f"  {c.text[:100]}...\n")

=== Multi-Topic Financial Retrieval ===

Q: What was the operating margin in FY2024?
  [0.587] [EC-Q4-2024] Forward Guidance (chunk 1/2) (Q4-2024)
  For FY2025, we are providing the following guidance:

Total revenue: $17.5 billion to $18.0 billion,...

Q: How fast is Cloud Services revenue growing?
  [0.749] [AR-2024] Revenue by Segment [TABLE] (chunk 1/1) (FY2024)
  Revenue by Business Segment (in millions USD):

| Segment | FY2024 Revenue | FY2023 Revenue | YoY Gr...

Q: What is the company's AI revenue?
  [0.738] [EC-Q4-2024] Analyst Q&A — AI Strategy (chunk 1/3) (Q4-2024)
  Analyst (Morgan Stanley): Can you give us more color on the AI revenue composition and what's drivin...

Q: What guidance did management give for FY2025?
  [0.495] [EC-Q4-2024] Forward Guidance (chunk 1/2) (Q4-2024)
  For FY2025, we are providing the following guidance:

Total revenue: $17.5 billion to $18.0 billion,...

Q: How much was spent on R&D?
  [0.578] [AR-2023] Research and Development (chunk 1/2) (FY2

In [11]:
# ── Filtered retrieval: tables only ──

print("=== Table-only search ===\n")
q = "What is the revenue breakdown by segment?"
results = fin_index.search(q, top_k=3, table_only=True)

for c, score in results:
    print(f"  [{score:.3f}] {c.citation}")
    print(f"  Period: {c.period}")
    print(f"  Text preview:\n{c.text[:300]}...")
    print()

=== Table-only search ===

  [0.498] [AR-2023] Revenue by Segment [TABLE] (chunk 1/1)
  Period: FY2023
  Text preview:
Revenue by Business Segment (in millions USD):

| Segment | FY2023 Revenue | FY2022 Revenue | YoY Growth |
|---------|---------------|---------------|------------|
| Cloud Services | $5,580 | $4,360 | 28.0% |
| Enterprise Software | $3,720 | $3,480 | 6.9% |
| Professional Services | $2,170 | $2,050 ...

  [0.485] [AR-2023] Profitability Metrics [TABLE] (chunk 1/1)
  Period: FY2023
  Text preview:
Key Profitability Metrics:

| Metric | FY2023 | FY2022 | Change |
|--------|--------|--------|--------|
| Gross Profit | $8,060M | $6,976M | +15.5% |
| Gross Margin | 65.0% | 64.0% | +1.0pp |
| Operating Income | $2,800M | $2,299M | +21.8% |
| Operating Margin | 22.6% | 21.1% | +1.5pp |
| Net Income...

  [0.475] [AR-2024] Profitability Metrics [TABLE] (chunk 1/1)
  Period: FY2024
  Text preview:
Key Profitability Metrics:

| Metric | FY2024 | FY2023 | Change |
|--------|-----

In [12]:
# ── Period-filtered search ──

print("=== FY2023 only ===\n")
q = "What was the revenue and net income?"
r2023 = fin_index.search(q, top_k=2, period_filter="FY2023")
for c, score in r2023:
    print(f"  [{score:.3f}] {c.citation} -> {c.numbers[:4]}")

print("\n=== FY2024 only ===\n")
r2024 = fin_index.search(q, top_k=2, period_filter="FY2024")
for c, score in r2024:
    print(f"  [{score:.3f}] {c.citation} -> {c.numbers[:4]}")

=== FY2023 only ===

  [0.486] [AR-2023] Profitability Metrics [TABLE] (chunk 1/1) -> ['$8,060M', '$6,976M', '$2,800M', '$2,299M']
  [0.471] [AR-2023] Financial Highlights (chunk 2/2) -> ['$2.1 billion', '$4.20 ', '$1.7 billion', '$3.40 ']

=== FY2024 only ===

  [0.490] [AR-2024] Profitability Metrics [TABLE] (chunk 1/1) -> ['$10,132M', '$8,060M', '$3,600M', '$2,800M']
  [0.471] [AR-2024] Financial Highlights (chunk 2/3) -> ['$2.7 billion', '$5.40 ', '$2.1 billion', '$4.20 ']


## Handling Numbers Carefully

Financial Q&A demands **numerical precision**. Our system:
1. Extracts numbers from retrieved chunks with their units and context
2. Validates that retrieved numbers match the query's time period
3. Presents exact figures rather than approximations
4. Warns when numbers from different periods might be confusing

In [13]:
# ── Number extraction and validation ──

def extract_financial_numbers(text):
    """Extract financial figures with context from text."""
    results = []

    # Dollar amounts with optional magnitude
    for match in re.finditer(
        r'(\$[\d,.]+)\s*(billion|million|B|M)?', text, re.IGNORECASE
    ):
        value_str = match.group(1)
        magnitude = (match.group(2) or "").lower()
        # Get surrounding context (20 chars before and after)
        start = max(0, match.start() - 40)
        end = min(len(text), match.end() + 40)
        context = text[start:end].replace("\n", " ").strip()
        results.append({
            "value": value_str,
            "magnitude": magnitude,
            "full": f"{value_str} {magnitude}".strip(),
            "context": context,
        })

    # Percentages
    for match in re.finditer(r'([\d,.]+)%', text):
        start = max(0, match.start() - 40)
        end = min(len(text), match.end() + 40)
        context = text[start:end].replace("\n", " ").strip()
        results.append({
            "value": match.group(0),
            "magnitude": "percent",
            "full": match.group(0),
            "context": context,
        })

    return results


# Demo: extract numbers from a revenue table chunk
table_chunks = [c for c in all_chunks if c.has_table]
sample = table_chunks[0]

print(f"Source: {sample.citation}\n")
numbers = extract_financial_numbers(sample.text)
print(f"Extracted {len(numbers)} financial figures:\n")
for n in numbers[:10]:
    print(f"  {n['full']:>20s}  <- {n['context'][:60]}")

Source: [AR-2023] Revenue by Segment [TABLE] (chunk 1/1)

Extracted 18 financial figures:

                $5,580  <- ------|------------| | Cloud Services | $5,580 | $4,360 | 28
                $4,360  <- ----------| | Cloud Services | $5,580 | $4,360 | 28.0% | | E
                $3,720  <- 4,360 | 28.0% | | Enterprise Software | $3,720 | $3,480 | 6.
                $3,480  <- 8.0% | | Enterprise Software | $3,720 | $3,480 | 6.9% | | Pr
                $2,170  <- ,480 | 6.9% | | Professional Services | $2,170 | $2,050 | 5.
                $2,050  <- 9% | | Professional Services | $2,170 | $2,050 | 5.9% | | Ha
                  $930  <- $2,050 | 5.9% | | Hardware Solutions | $930 | $1,010 | -7.9%
                $1,010  <- | 5.9% | | Hardware Solutions | $930 | $1,010 | -7.9% | | To
               $12,400  <- ons | $930 | $1,010 | -7.9% | | Total | $12,400 | $10,900 | 
               $10,900  <- | $1,010 | -7.9% | | Total | $12,400 | $10,900 | 13.8% |  Cl


## Full Financial Report Analyst Pipeline

In [14]:
# ── Financial Report Analyst ──

class FinancialAnalyst:
    """Financial document Q&A with table awareness and number precision."""

    def __init__(self, index):
        self.index = index

    def ask(self, question, top_k=TOP_K, period_filter=None,
            doc_type_filter=None, table_only=False, verbose=True):
        """Ask a financial question and get a grounded answer."""
        results = self.index.search(
            question, top_k=top_k,
            period_filter=period_filter,
            doc_type_filter=doc_type_filter,
            table_only=table_only,
        )

        if not results:
            return {"answer": "No relevant information found.", "sources": [],
                    "numbers": [], "confidence": 0.0}

        answer = self._synthesize(question, results)
        numbers = self._extract_key_numbers(question, results)
        confidence = self._confidence([s for _, s in results])

        sources = [{
            "citation": c.citation,
            "period": c.period,
            "score": round(s, 3),
            "has_table": c.has_table,
        } for c, s in results[:3]]

        if verbose:
            self._display(question, answer, sources, numbers, confidence,
                          period_filter, table_only)

        return {"answer": answer, "sources": sources, "numbers": numbers,
                "confidence": confidence}

    def _synthesize(self, question, results):
        q_terms = set(question.lower().split())
        parts = []
        for chunk, score in results[:3]:
            sentences = re.split(r'(?<=[.!?])\s+', chunk.text)
            relevant = []
            for sent in sentences:
                # Also boost sentences with numbers for financial queries
                has_number = bool(re.search(r'\$|\d+%|\d+\.\d+', sent))
                overlap = sum(1 for t in q_terms if t in sent.lower())
                if (overlap > 0 or has_number) and len(sent) > 15:
                    relevant.append((sent, overlap + (2 if has_number else 0)))
            relevant.sort(key=lambda x: x[1], reverse=True)
            best = [s for s, _ in relevant[:3]]
            if best:
                parts.append(f"{' '.join(best)} [{chunk.doc_id}, {chunk.heading}]")
        if parts:
            return "\n\n".join(parts)
        return f"{results[0][0].text[:300]}... [{results[0][0].doc_id}]"

    def _extract_key_numbers(self, question, results):
        all_nums = []
        for chunk, _ in results[:3]:
            nums = extract_financial_numbers(chunk.text)
            for n in nums:
                n["source_period"] = chunk.period
                n["source_doc"] = chunk.doc_id
            all_nums.extend(nums)
        return all_nums[:10]

    def _confidence(self, scores):
        top = scores[0]
        avg = np.mean(scores) if scores else 0
        conf = 0.6 * min(top / 0.7, 1.0) + 0.4 * min(avg / 0.5, 1.0)
        return round(min(conf, 1.0), 3)

    def _display(self, question, answer, sources, numbers, confidence,
                 period_filter, table_only):
        print("=" * 70)
        print(f"Q: {question}")
        filters = []
        if period_filter:
            filters.append(f"period={period_filter}")
        if table_only:
            filters.append("tables_only")
        if filters:
            print(f"   Filters: {', '.join(filters)}")
        print("=" * 70)

        print(f"\nAnswer:")
        for line in answer.split("\n"):
            if line.strip():
                wrapped = textwrap.fill(line.strip(), width=66,
                                        initial_indent="  ", subsequent_indent="  ")
                print(wrapped)

        level = "HIGH" if confidence >= 0.7 else ("MEDIUM" if confidence >= 0.4 else "LOW")
        bar = "#" * int(confidence * 25)
        print(f"\nConfidence: {confidence:.3f} ({level}) |{bar}|")

        if numbers:
            print(f"\nKey Numbers Found:")
            seen = set()
            for n in numbers[:6]:
                key = n["full"]
                if key not in seen:
                    print(f"  {n['full']:>20s}  ({n['source_period']}) "
                          f"<- ...{n['context'][:45]}...")
                    seen.add(key)

        print(f"\nSources:")
        for i, src in enumerate(sources, 1):
            tbl = " [TABLE]" if src["has_table"] else ""
            print(f"  [{i}] {src['citation']}{tbl} (score: {src['score']:.3f})")

        print("=" * 70)


analyst = FinancialAnalyst(fin_index)
print("Financial Report Analyst ready.")

Financial Report Analyst ready.


## Query Examples

In [15]:
# ── Query 1: Revenue ──
analyst.ask("What was TechGlobal's total revenue in FY2024?")

Q: What was TechGlobal's total revenue in FY2024?

Answer:
  achieved record financial performance in fiscal year 2024, with
  total revenue of $14.9 billion, representing 20% year-over-year
  growth from FY2023's $12.4 billion. Operating income reached
  $3.6 billion, a 29% increase from $2.8 billion in FY2023.
  Operating margin expanded to 24.2%, up from 22.6%, driven by AI
  product premiums and continued operational leverage. [AR-2024,
  Financial Highlights]
  Total revenue reached $12.4 billion, representing a 14% increase
  from the prior year's $10.9 billion. The growth was driven
  primarily by our Cloud Services division, which saw 28% year-
  over-year revenue growth. Operating income expanded to $2.8
  billion, up from $2.3 billion in FY2022, reflecting a 22%
  improvement. [AR-2023, Financial Highlights]
  For FY2025, we are providing the following guidance:
  Total revenue: $17.5 billion to $18.0 billion, representing
  17-21% year-over-year growth. This includes approxi

{'answer': "achieved record financial performance in fiscal year 2024, with total revenue of $14.9 billion, representing 20% year-over-year growth from FY2023's $12.4 billion. Operating income reached $3.6 billion, a 29% increase from $2.8 billion in FY2023. Operating margin expanded to 24.2%, up from 22.6%, driven by AI product premiums and continued operational leverage. [AR-2024, Financial Highlights]\n\nTotal revenue reached $12.4 billion, representing a 14% increase from the prior year's $10.9 billion. The growth was driven primarily by our Cloud Services division, which saw 28% year-over-year revenue growth. Operating income expanded to $2.8 billion, up from $2.3 billion in FY2022, reflecting a 22% improvement. [AR-2023, Financial Highlights]\n\nFor FY2025, we are providing the following guidance:\n\nTotal revenue: $17.5 billion to $18.0 billion, representing 17-21% year-over-year growth. This includes approximately $600 million in expected contribution from the DataSphere and Se

In [16]:
# ── Query 2: Margins ──
analyst.ask("How did operating margin change from FY2023 to FY2024?")


Q: How did operating margin change from FY2023 to FY2024?

Answer:
  Operating margin: 25.0% to 26.0%, reflecting continued leverage
  and mix improvement. We expect gross margin to reach
  approximately 69-70%. Diluted EPS: $6.30 to $6.60, representing
  17-22% growth from FY2024's $5.40. [EC-Q4-2024, Forward
  Guidance]
  Key Profitability Metrics:
  | Metric | FY2024 | FY2023 | Change |
  |--------|--------|--------|--------|
  | Gross Profit | $10,132M | $8,060M | +25.7% |
  | Gross Margin | 68.0% | 65.0% | +3.0pp |
  | Operating Income | $3,600M | $2,800M | +28.6% |
  | Operating Margin | 24.2% | 22.6% | +1.6pp |
  | Net Income | $2,700M | $2,100M | +28.6% |
  | Net Margin | 18.1% | 16.9% | +1.2pp |
  | EBITDA | $4,500M | $3,500M | +28.6% |
  | EBITDA Margin | 30.2% | 28.2% | +2.0pp |
  | Diluted EPS | $5.40 | $4.20 | +28.6% |
  | Free Cash Flow | $4,100M | $3,200M | +28.1% |
  Gross margin expansion to 68% reflects the growing mix of high-
  margin AI and cloud subscription reve

{'answer': "Operating margin: 25.0% to 26.0%, reflecting continued leverage and mix improvement. We expect gross margin to reach approximately 69-70%. Diluted EPS: $6.30 to $6.60, representing 17-22% growth from FY2024's $5.40. [EC-Q4-2024, Forward Guidance]\n\nKey Profitability Metrics:\n\n| Metric | FY2024 | FY2023 | Change |\n|--------|--------|--------|--------|\n| Gross Profit | $10,132M | $8,060M | +25.7% |\n| Gross Margin | 68.0% | 65.0% | +3.0pp |\n| Operating Income | $3,600M | $2,800M | +28.6% |\n| Operating Margin | 24.2% | 22.6% | +1.6pp |\n| Net Income | $2,700M | $2,100M | +28.6% |\n| Net Margin | 18.1% | 16.9% | +1.2pp |\n| EBITDA | $4,500M | $3,500M | +28.6% |\n| EBITDA Margin | 30.2% | 28.2% | +2.0pp |\n| Diluted EPS | $5.40 | $4.20 | +28.6% |\n| Free Cash Flow | $4,100M | $3,200M | +28.1% |\n\nGross margin expansion to 68% reflects the growing mix of high-margin AI and cloud subscription revenue. EBITDA margin crossed 30% for the first time in company history. [AR-202

In [17]:
# ── Query 3: Segment breakdown ──
analyst.ask("What is the Cloud Services revenue and growth rate?", table_only=True)

Q: What is the Cloud Services revenue and growth rate?
   Filters: tables_only

Answer:
  Revenue by Business Segment (in millions USD):
  | Segment | FY2024 Revenue | FY2023 Revenue | YoY Growth |
  |---------|---------------|---------------|------------|
  | Cloud Services | $7,450 | $5,580 | 33.5% |
  | Enterprise Software | $4,030 | $3,720 | 8.3% |
  | Professional Services | $2,520 | $2,170 | 16.1% |
  | Hardware Solutions | $900 | $930 | -3.2% |
  | Total | $14,900 | $12,400 | 20.2% |
  Cloud Services now represents 50% of total revenue, crossing the
  majority threshold for the first time. AI-powered products
  within Cloud Services grew 85% year-over-year and now represent
  $2.2 billion in annual recurring revenue. Professional Services
  growth accelerated due to increased demand for AI implementation
  consulting. [AR-2024, Revenue by Segment]
  Revenue by Business Segment (in millions USD):
  | Segment | FY2023 Revenue | FY2022 Revenue | YoY Growth |
  |---------|----------

{'answer': 'Revenue by Business Segment (in millions USD):\n\n| Segment | FY2024 Revenue | FY2023 Revenue | YoY Growth |\n|---------|---------------|---------------|------------|\n| Cloud Services | $7,450 | $5,580 | 33.5% |\n| Enterprise Software | $4,030 | $3,720 | 8.3% |\n| Professional Services | $2,520 | $2,170 | 16.1% |\n| Hardware Solutions | $900 | $930 | -3.2% |\n| Total | $14,900 | $12,400 | 20.2% |\n\nCloud Services now represents 50% of total revenue, crossing the majority threshold for the first time. AI-powered products within Cloud Services grew 85% year-over-year and now represent $2.2 billion in annual recurring revenue. Professional Services growth accelerated due to increased demand for AI implementation consulting. [AR-2024, Revenue by Segment]\n\nRevenue by Business Segment (in millions USD):\n\n| Segment | FY2023 Revenue | FY2022 Revenue | YoY Growth |\n|---------|---------------|---------------|------------|\n| Cloud Services | $5,580 | $4,360 | 28.0% |\n| Enterp

In [18]:
# ── Query 4: Forward guidance ──
analyst.ask("What revenue guidance did management give for FY2025?",
            doc_type_filter="earnings_call")

Q: What revenue guidance did management give for FY2025?

Answer:
  For FY2025, we are providing the following guidance:
  Total revenue: $17.5 billion to $18.0 billion, representing
  17-21% year-over-year growth. Capital expenditures: $2.0 billion
  to $2.2 billion, primarily for data center e This includes
  approximately $600 million in expected contribution from the
  DataSphere and SecureNet acquisitions. [EC-Q4-2024, Forward
  Guidance]
  Q4 revenue of $4.1 billion exceeded our guidance range of $3.9
  to $4.0 billion. Full-year revenue was $14.9 billion, up 20%
  year-over-year. On a constant currency basis, growth was
  approximately 21%. [EC-Q4-2024, CFO Financial Review]
  Q4 revenue was $4.1 billion, up 22% year-over-year, with Cloud
  Services growing 35% in the quarter. Our AI annual recurring
  revenue exited the year at $2.2 billion, and we see a clear path
  to $3.5 billion by the end of FY2025. We signed over 200 new
  enterprise AI deals in Q4 alone, with an average 

{'answer': 'For FY2025, we are providing the following guidance:\n\nTotal revenue: $17.5 billion to $18.0 billion, representing 17-21% year-over-year growth. Capital expenditures: $2.0 billion to $2.2 billion, primarily for data center e This includes approximately $600 million in expected contribution from the DataSphere and SecureNet acquisitions. [EC-Q4-2024, Forward Guidance]\n\nQ4 revenue of $4.1 billion exceeded our guidance range of $3.9 to $4.0 billion. Full-year revenue was $14.9 billion, up 20% year-over-year. On a constant currency basis, growth was approximately 21%. [EC-Q4-2024, CFO Financial Review]\n\nQ4 revenue was $4.1 billion, up 22% year-over-year, with Cloud Services growing 35% in the quarter. Our AI annual recurring revenue exited the year at $2.2 billion, and we see a clear path to $3.5 billion by the end of FY2025. We signed over 200 new enterprise AI deals in Q4 alone, with an average contract value of $2.8 million, up 40% from Q4 last year. [EC-Q4-2024, CEO Op

In [19]:
# ── Query 5: AI strategy ──
analyst.ask("What is TechGlobal's AI revenue and how fast is it growing?")

Q: What is TechGlobal's AI revenue and how fast is it growing?

Answer:
  achieved record financial performance in fiscal year 2024, with
  total revenue of $14.9 billion, representing 20% year-over-year
  growth from FY2023's $12.4 billion. Operating margin expanded to
  24.2%, up from 22.6%, driven by AI product premiums and
  continued operational leverage. This acceleration was fueled by
  exceptional demand for AI-powered cloud solutions and successful
  expansion into new geographic markets. [AR-2024, Financial
  Highlights]
  Our AI revenue of $2.2 billion breaks down roughly as follows:
  about 55% comes from our AI-as-a-Service platform, which is
  essentially our managed AI infrastructure and model hosting.
  About 30% comes from AI-embedded features within our existing
  Enterprise Software products — things like intelligent
  automation, predictive analytics, and AI-powered security.
  Analyst (Morgan Stanley): Can you give us more color on the AI
  revenue composition and 

{'answer': "achieved record financial performance in fiscal year 2024, with total revenue of $14.9 billion, representing 20% year-over-year growth from FY2023's $12.4 billion. Operating margin expanded to 24.2%, up from 22.6%, driven by AI product premiums and continued operational leverage. This acceleration was fueled by exceptional demand for AI-powered cloud solutions and successful expansion into new geographic markets. [AR-2024, Financial Highlights]\n\nOur AI revenue of $2.2 billion breaks down roughly as follows: about 55% comes from our AI-as-a-Service platform, which is essentially our managed AI infrastructure and model hosting. About 30% comes from AI-embedded features within our existing Enterprise Software products — things like intelligent automation, predictive analytics, and AI-powered security. Analyst (Morgan Stanley): Can you give us more color on the AI revenue composition and what's driving the acceleration? [EC-Q4-2024, Analyst Q&A — AI Strategy]\n\nWe expect AI 

In [20]:
# ── Query 6: Year-over-year comparison ──
print("=== FY2023 ===")
analyst.ask("What was the diluted EPS?", period_filter="FY2023")
print("\n=== FY2024 ===")
analyst.ask("What was the diluted EPS?", period_filter="FY2024")

=== FY2023 ===
Q: What was the diluted EPS?
   Filters: period=FY2023

Answer:
  Key Profitability Metrics:
  | Metric | FY2023 | FY2022 | Change |
  |--------|--------|--------|--------|
  | Gross Profit | $8,060M | $6,976M | +15.5% |
  | Gross Margin | 65.0% | 64.0% | +1.0pp |
  | Operating Income | $2,800M | $2,299M | +21.8% |
  | Operating Margin | 22.6% | 21.1% | +1.5pp |
  | Net Income | $2,100M | $1,700M | +23.5% |
  | Net Margin | 16.9% | 15.6% | +1.3pp |
  | EBITDA | $3,500M | $2,950M | +18.6% |
  | EBITDA Margin | 28.2% | 27.1% | +1.1pp |
  | Diluted EPS | $4.20 | $3.40 | +23.5% |
  | Free Cash Flow | $3,200M | $2,700M | +18.5% |
  All margins showed improvement year-over-year, reflecting the
  company's successful transition toward higher-margin recurring
  revenue streams and disciplined cost management. [AR-2023,
  Profitability Metrics]
  Net income attributable to shareholders was $2.1 billion, or
  $4.20 per diluted share, compared to $1.7 billion, or $3.40 per
  dilute

{'answer': 'Net income grew to $2.7 billion, or $5.40 per diluted share, compared to $2.1 billion, or $4.20 per diluted share, in FY2023 — a 29% increase in earnings per share. The company completed two strategic acquisitions: DataSphere (cloud data analytics, $1.2B) and SecureNet (AI-powered cybersecurity, $800M), which are expected to contribute $600M in incremental revenue in FY2025. Free cash flow reached $4.1 billion, providing ample capacity for both organic investment and M&A activity. [AR-2024, Financial Highlights]\n\nKey Profitability Metrics:\n\n| Metric | FY2024 | FY2023 | Change |\n|--------|--------|--------|--------|\n| Gross Profit | $10,132M | $8,060M | +25.7% |\n| Gross Margin | 68.0% | 65.0% | +3.0pp |\n| Operating Income | $3,600M | $2,800M | +28.6% |\n| Operating Margin | 24.2% | 22.6% | +1.6pp |\n| Net Income | $2,700M | $2,100M | +28.6% |\n| Net Margin | 18.1% | 16.9% | +1.2pp |\n| EBITDA | $4,500M | $3,500M | +28.6% |\n| EBITDA Margin | 30.2% | 28.2% | +2.0pp |\

## Year-over-Year Comparison

A common financial analysis task is comparing metrics across years.
Let's build a simple comparison tool.

In [21]:
# ── YoY comparison ──

def compare_periods(question, index, periods):
    """Compare retrieval results across different periods."""
    print(f"Question: {question}")
    print(f"Comparing: {' vs '.join(periods)}")
    print("-" * 60)

    for period in periods:
        results = index.search(question, top_k=2, period_filter=period)
        if results:
            chunk, score = results[0]
            nums = extract_financial_numbers(chunk.text)
            key_nums = [n["full"] for n in nums[:5]]
            print(f"\n  [{period}] (score: {score:.3f})")
            print(f"  Source: {chunk.citation}")
            print(f"  Key figures: {', '.join(key_nums[:4])}")
            print(f"  Excerpt: {chunk.text[:150]}...")

    print("-" * 60)


compare_periods("What was the total revenue and net income?",
                fin_index, ["FY2023", "FY2024"])

Question: What was the total revenue and net income?
Comparing: FY2023 vs FY2024
------------------------------------------------------------

  [FY2023] (score: 0.514)
  Source: [AR-2023] Profitability Metrics [TABLE] (chunk 1/1)
  Key figures: $8,060 m, $6,976 m, $2,800 m, $2,299 m
  Excerpt: Key Profitability Metrics:

| Metric | FY2023 | FY2022 | Change |
|--------|--------|--------|--------|
| Gross Profit | $8,060M | $6,976M | +15.5% |
...

  [FY2024] (score: 0.520)
  Source: [AR-2024] Profitability Metrics [TABLE] (chunk 1/1)
  Key figures: $10,132 m, $8,060 m, $3,600 m, $2,800 m
  Excerpt: Key Profitability Metrics:

| Metric | FY2024 | FY2023 | Change |
|--------|--------|--------|--------|
| Gross Profit | $10,132M | $8,060M | +25.7% |...
------------------------------------------------------------


In [22]:
compare_periods("What was the gross margin and operating margin?",
                fin_index, ["FY2023", "FY2024"])

Question: What was the gross margin and operating margin?
Comparing: FY2023 vs FY2024
------------------------------------------------------------

  [FY2023] (score: 0.626)
  Source: [AR-2023] Profitability Metrics [TABLE] (chunk 1/1)
  Key figures: $8,060 m, $6,976 m, $2,800 m, $2,299 m
  Excerpt: Key Profitability Metrics:

| Metric | FY2023 | FY2022 | Change |
|--------|--------|--------|--------|
| Gross Profit | $8,060M | $6,976M | +15.5% |
...



  [FY2024] (score: 0.611)
  Source: [AR-2024] Profitability Metrics [TABLE] (chunk 1/1)
  Key figures: $10,132 m, $8,060 m, $3,600 m, $2,800 m
  Excerpt: Key Profitability Metrics:

| Metric | FY2024 | FY2023 | Change |
|--------|--------|--------|--------|
| Gross Profit | $10,132M | $8,060M | +25.7% |...
------------------------------------------------------------


## Evaluation

In [23]:
# ── Evaluation ──

eval_set = [
    {"query": "Total revenue FY2024",
     "expected_doc": "AR-2024", "expected_type": "table"},
    {"query": "Operating margin improvement",
     "expected_doc": "AR-2024", "expected_type": "table"},
    {"query": "Cloud Services growth rate",
     "expected_doc": "AR-2024", "expected_type": "table"},
    {"query": "FY2025 revenue guidance",
     "expected_doc": "EC-Q4-2024", "expected_type": "narrative"},
    {"query": "R&D investment and patents",
     "expected_doc": "AR-2023", "expected_type": "narrative"},
    {"query": "Cash and equivalents on balance sheet",
     "expected_doc": "AR-2023", "expected_type": "table"},
    {"query": "AI annual recurring revenue",
     "expected_doc": "EC-Q4-2024", "expected_type": "narrative"},
    {"query": "Geographic revenue breakdown",
     "expected_doc": "AR-2024", "expected_type": "table"},
    {"query": "Employee headcount",
     "expected_doc": "AR-2024", "expected_type": "narrative"},
    {"query": "Share repurchase program",
     "expected_doc": "EC-Q4-2024", "expected_type": "narrative"},
]

baseline_hits = 0
semantic_hits = 0

print(f"{'Query':<40} {'Baseline':^10} {'Semantic':^10}")
print("-" * 60)

for e in eval_set:
    q = e["query"]
    expected = e["expected_doc"]

    b = keyword_search(q, all_chunks, top_k=1)
    b_hit = b[0][0].doc_id == expected if b else False
    baseline_hits += int(b_hit)

    s = fin_index.search(q, top_k=1)
    s_hit = s[0][0].doc_id == expected if s else False
    semantic_hits += int(s_hit)

    print(f"  {q:<38} {'HIT' if b_hit else 'MISS':^10} {'HIT' if s_hit else 'MISS':^10}")

n = len(eval_set)
print("-" * 60)
print(f"  {'ACCURACY':<38} {baseline_hits/n:^10.1%} {semantic_hits/n:^10.1%}")

Query                                     Baseline   Semantic 
------------------------------------------------------------
  Total revenue FY2024                      HIT        MISS   
  Operating margin improvement              MISS       MISS   
  Cloud Services growth rate                HIT        HIT    


  FY2025 revenue guidance                   HIT        HIT    
  R&D investment and patents                HIT        HIT    


  Cash and equivalents on balance sheet     HIT        HIT    
  AI annual recurring revenue               MISS       MISS   


  Geographic revenue breakdown              HIT        HIT    
  Employee headcount                        HIT        HIT    


  Share repurchase program                  HIT        HIT    
------------------------------------------------------------
  ACCURACY                                 80.0%      70.0%   


## Error Analysis

In [24]:
# ── Edge cases ──

edge_queries = [
    "What was the stock price?",              # Not in our documents
    "Compare TechGlobal to competitors",      # No competitor data
    "Is TechGlobal a good investment?",        # Opinion, not fact
    "EBITDA margin trend over 5 years",       # Only 2 years of data
]

print("=== Edge Cases ===\n")
for q in edge_queries:
    results = fin_index.search(q, top_k=2)
    top_score = results[0][1] if results else 0
    print(f"Q: \"{q}\"")
    print(f"  Top score: {top_score:.3f}")
    if top_score < 0.3:
        print(f"  -> Low confidence: question may be out of scope")
    else:
        print(f"  -> Retrieved: {results[0][0].citation}")
    print()

=== Edge Cases ===

Q: "What was the stock price?"
  Top score: 0.373
  -> Retrieved: [EC-Q4-2024] CFO Financial Review (chunk 2/2)



Q: "Compare TechGlobal to competitors"
  Top score: 0.453
  -> Retrieved: [AR-2023] Financial Highlights (chunk 1/2)

Q: "Is TechGlobal a good investment?"
  Top score: 0.577
  -> Retrieved: [AR-2023] Financial Highlights (chunk 1/2)

Q: "EBITDA margin trend over 5 years"
  Top score: 0.441
  -> Retrieved: [EC-Q4-2024] CFO Financial Review (chunk 1/2)



In [25]:
# ── Export metrics ──

metrics = {
    "project": "Financial Report Analyst",
    "corpus": {
        "documents": len(FINANCIAL_DOCUMENTS),
        "sections": len(all_sections),
        "chunks": len(all_chunks),
        "table_chunks": len([c for c in all_chunks if c.has_table]),
        "narrative_chunks": len([c for c in all_chunks if not c.has_table]),
        "periods": list(set(c.period for c in all_chunks)),
    },
    "retrieval_backend": fin_index.backend,
    "evaluation": {
        "total_queries": len(eval_set),
        "baseline_accuracy": baseline_hits / len(eval_set),
        "semantic_accuracy": semantic_hits / len(eval_set),
    },
    "config": {
        "chunk_size": CHUNK_SIZE,
        "chunk_overlap": CHUNK_OVERLAP,
        "top_k": TOP_K,
        "embedding_model": EMBEDDING_MODEL if USE_ST else "TF-IDF",
    },
}

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\GenAI\14_Financial_Report_Analyst\metrics.json
{
  "project": "Financial Report Analyst",
  "corpus": {
    "documents": 3,
    "sections": 15,
    "chunks": 26,
    "table_chunks": 6,
    "narrative_chunks": 20,
    "periods": [
      "FY2023",
      "FY2024",
      "Q4-2024"
    ]
  },
  "retrieval_backend": "chroma",
  "evaluation": {
    "total_queries": 10,
    "baseline_accuracy": 0.8,
    "semantic_accuracy": 0.7
  },
  "config": {
    "chunk_size": 600,
    "chunk_overlap": 100,
    "top_k": 5,
    "embedding_model": "all-MiniLM-L6-v2"
  }
}


## Limitations

1. **No real PDF parsing** — we use pre-structured text. Real annual reports
   require PDF extraction (PyMuPDF, pdfplumber, Unstructured.io) with layout
   detection for tables.

2. **No cross-document reasoning** — "How did margin compare across years?"
   requires combining data from two documents, which single-query
   retrieval doesn't do well.

3. **No numerical computation** — the system retrieves numbers but can't
   compute growth rates, ratios, or aggregations itself. An LLM or
   code execution layer would be needed.

4. **Limited table understanding** — markdown tables are handled, but complex
   multi-header tables, footnotes, and merged cells from real filings
   would require more sophisticated parsing.

5. **No XBRL / structured data** — SEC filings include machine-readable XBRL
   tags. A production system would parse XBRL for exact numerical data.

6. **Small corpus** — real financial analysis spans hundreds of filings across
   multiple companies and years.

## Common Mistakes

1. **Destroying tables during chunking** — splitting a table row across chunks
   makes both chunks useless. Always keep table rows together with headers.

2. **Mixing up time periods** — retrieving FY2023 data for a FY2024 question.
   Always track and display the period metadata prominently.

3. **Losing numerical precision** — rounding "$14.9 billion" to "~$15 billion"
   or dropping units. In finance, precision matters.

4. **Ignoring table vs narrative context** — a question about "revenue breakdown"
   should prioritize table chunks. A question about "management strategy" should
   prioritize narrative chunks. Content-type filtering helps.

5. **Not validating extracted numbers** — just because a number appears in a
   retrieved chunk doesn't mean it answers the question. A chunk about revenue
   might also mention headcount numbers.

6. **Treating earnings calls like annual reports** — earnings calls contain
   forward-looking statements and management opinions, not just facts.
   The system should note when answers come from forward guidance vs actuals.

## Mini Challenge

### Exercise 1: Add Q1 2025 Earnings Data
Create a fourth document with Q1 2025 results based on the guidance numbers.
Re-index and test queries that span Q4 2024 and Q1 2025.

### Exercise 2: Numerical Comparison
Build a function that retrieves the same metric from two periods and
computes the year-over-year change automatically:
```python
def compute_yoy(metric_query, index, period1, period2):
    # Retrieve numbers from both periods
    # Parse dollar amounts
    # Compute change
    pass
```

### Exercise 3: Segment-Level Deep Dive
Write queries that zoom into one segment (e.g., Cloud Services) across
both years. Compute the segment's revenue CAGR (compound annual growth rate).

### Exercise 4: Table Extraction
Build a function that parses a markdown table from chunk text into a
list of dictionaries for programmatic access.

### Exercise 5: Earnings Call Sentiment
Analyze the CEO's language in the earnings call. Count positive vs negative
words. Does the tone match the actual financial results?

## Production Considerations

1. **PDF Parsing Pipeline** — Use Unstructured.io, DocTR, or Azure Document
   Intelligence for extracting text and tables from real SEC filings.

2. **XBRL Integration** — Parse SEC XBRL filings for machine-readable
   financial data. Libraries: `python-xbrl`, SEC EDGAR API.

3. **Financial LLM** — Use a finance-tuned LLM (FinGPT, BloombergGPT) or
   GPT-4 with financial system prompts for answer generation.

4. **Time-Series Database** — Store historical financial data in a time-series
   DB (InfluxDB, TimescaleDB) for fast numerical queries.

5. **Multi-Company Support** — Scale to index filings from multiple companies
   with company-level metadata filtering.

6. **Real-Time Updates** — Monitor SEC EDGAR for new filings and automatically
   ingest them. RSS feeds or EDGAR full-text search API.

7. **Compliance** — Ensure no material non-public information (MNPI) leaks.
   Add access controls and audit trails.

8. **Visualization** — Generate charts from extracted data (revenue trends,
   margin evolution) using matplotlib or plotly.

## How to Improve This Project

- **Structured table extraction** — Parse markdown tables into DataFrames for
  programmatic queries ("what was the fastest growing segment?").
- **Cross-period reasoning** — Build a pipeline that retrieves from multiple
  periods and computes deltas automatically.
- **Financial formula engine** — Add computation capabilities (growth rate,
  CAGR, margin calculations) on top of retrieved numbers.
- **Fine-tuned embeddings** — Train on financial text (10-K filings, earnings
  transcripts) for better domain-specific retrieval.
- **Chart generation** — Auto-generate comparison charts from retrieved data.
- **Alert system** — Flag significant changes (margin decline, revenue miss
  vs guidance) automatically.

## Key Takeaways

1. **Tables require special treatment** — never split table rows across chunks.
   Always preserve headers and structure. Financial data lives in tables.

2. **Numbers need precision** — extract and display exact figures with units.
   Financial Q&A tolerates zero rounding error.

3. **Time period tracking is essential** — every chunk must carry its period
   metadata. Mixing FY2023 and FY2024 data silently is a critical failure.

4. **Content-type filtering improves precision** — "revenue by segment" should
   search tables; "management strategy" should search narrative text.

5. **Financial RAG needs domain awareness** — understanding terms like EBITDA,
   diluted EPS, and free cash flow is necessary for accurate retrieval.

6. **Baseline comparison matters** — if vector search doesn't beat keyword
   search for numerical queries, the embedding model may need fine-tuning.